# Rabbit v1 — Fine-tuning

Fine-tunes Phi-3.5 Mini (3.8B) on 55,750 examples across 8 memory signals.

**Requirements:** GPU runtime (T4 minimum, A100 recommended)

**Time:** ~2-3 hours on T4, ~45 min on A100

**Cost:** Free on Colab (T4) or ~$2-3 on Colab Pro (A100)

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

## Step 2: Upload Training Data

Upload your `data/filtered/` folder. Run this cell, then use the file picker.

In [ ]:
import os

# Option A: Upload from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/rabbit/data/filtered'

# Option B: Upload directly (run this, then upload files)
from google.colab import files
os.makedirs('data/filtered', exist_ok=True)

print("Upload all *_filtered.jsonl files from data/filtered/")
print("Files needed:")
for task in ['intent', 'extract', 'triage', 'expand', 'answer', 'summarize', 'sentiment', 'importance']:
    print(f"  - {task}_filtered.jsonl")

uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, f'data/filtered/{fname}')
    print(f"  Saved: data/filtered/{fname}")

DATA_DIR = 'data/filtered'

In [ ]:
# Verify data
import json
from pathlib import Path

DATA_DIR = Path('data/filtered')
TASKS = ['intent', 'extract', 'triage', 'expand', 'answer', 'summarize', 'sentiment', 'importance']

total = 0
for task in TASKS:
    f = DATA_DIR / f'{task}_filtered.jsonl'
    if f.exists():
        count = sum(1 for line in open(f) if line.strip())
        print(f'  {task:15s} {count:>8,} examples')
        total += count
    else:
        print(f'  {task:15s}  MISSING')

print(f'  {"TOTAL":15s} {total:>8,} examples')

## Step 3: Load Base Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# Phi-3.5 Mini — 3.8B parameters, excellent at structured tasks
BASE_MODEL = "unsloth/Phi-3.5-mini-instruct"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect (float16 on T4, bfloat16 on A100)
    load_in_4bit=True,  # QLoRA — fits in 8GB VRAM
)

print(f"Model loaded: {BASE_MODEL}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Step 4: Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 60% less VRAM
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total_params:,} ({trainable/total_params*100:.1f}%)")

## Step 5: Prepare Training Data

In [ ]:
from datasets import Dataset

# Task configuration
TASK_PREFIXES = {
    'intent': '[INTENT]',
    'extract': '[EXTRACT]',
    'triage': '[TRIAGE]',
    'expand': '[EXPAND]',
    'answer': '[ANSWER]',
    'summarize': '[SUMMARIZE]',
    'sentiment': '[SENTIMENT]',
    'importance': '[IMPORTANCE]',
}

TASK_SYSTEM_PROMPTS = {
    'intent': 'You are Rabbit, Reattend\'s memory AI. Classify the user\'s query intent. Respond with exactly one word: factual, entity, temporal, synthesis, actions, history, or aggregation.',
    'extract': 'You are Rabbit, Reattend\'s memory AI. Extract structured information from the given text. Return a JSON object with keys: people, organizations, decisions, action_items, dates, topics.',
    'triage': 'You are Rabbit, Reattend\'s memory AI. Classify and summarize the given content. Return a JSON object with keys: type, summary, tags.',
    'expand': 'You are Rabbit, Reattend\'s memory AI. Expand the user\'s vague query into a precise, comprehensive search query that captures their likely intent.',
    'answer': 'You are Rabbit, Reattend\'s memory AI. Answer the user\'s question using the provided memory context. Use citations [1][2][3] to reference sources. Do not use markdown formatting.',
    'summarize': 'You are Rabbit, Reattend\'s memory AI. Generate a rich 2-4 sentence standalone summary of the given content. Capture the essence, key decisions, and action items.',
    'sentiment': 'You are Rabbit, Reattend\'s memory AI. Classify the tone of the given content. Respond with exactly one word: positive, negative, neutral, tense, or urgent.',
    'importance': 'You are Rabbit, Reattend\'s memory AI. Score the importance of the given content for organizational memory. Return a JSON object with keys: score (1-5) and reason (one sentence).',
}

# Load all filtered data
all_examples = []

for task in TASKS:
    filepath = DATA_DIR / f'{task}_filtered.jsonl'
    if not filepath.exists():
        print(f'  Skipping {task} (not found)')
        continue

    count = 0
    with open(filepath) as f:
        for line in f:
            if not line.strip():
                continue
            raw = json.loads(line.strip())

            output = raw['output']
            if isinstance(output, dict):
                output = json.dumps(output)

            example = {
                'conversations': [
                    {'role': 'system', 'content': TASK_SYSTEM_PROMPTS[task]},
                    {'role': 'user', 'content': f"{TASK_PREFIXES[task]} {raw['input']}"},
                    {'role': 'assistant', 'content': output},
                ]
            }
            all_examples.append(example)
            count += 1

    print(f'  {task}: {count} examples')

print(f'\n  Total: {len(all_examples)} training examples')

# Format for tokenizer
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example['conversations'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

dataset = Dataset.from_list(all_examples)
dataset = dataset.map(format_chat)
dataset = dataset.shuffle(seed=42)

# Split 95/5
split = dataset.train_test_split(test_size=0.05, seed=42)
print(f'  Train: {len(split["train"]):,}')
print(f'  Eval:  {len(split["test"]):,}')

## Step 6: Train Rabbit v1

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_PATH = 'rabbit-v1'

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        output_dir=OUTPUT_PATH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy='steps',
        eval_steps=200,
        save_strategy='steps',
        save_steps=500,
        save_total_limit=3,
        report_to='none',
        optim='adamw_8bit',
    ),
)

print('Starting training...\n')
trainer_stats = trainer.train()

print(f'\nTraining complete!')
print(f'  Total steps: {trainer_stats.global_step}')
print(f'  Training loss: {trainer_stats.training_loss:.4f}')
print(f'  Runtime: {trainer_stats.metrics["train_runtime"]/60:.1f} minutes')

## Step 7: Test Rabbit v1

In [ ]:
# Quick smoke test on all 8 signals
FastLanguageModel.for_inference(model)

test_cases = [
    ('[INTENT]', 'What did we discuss with Brian last week?'),
    ('[EXTRACT]', 'Met with Sarah from Acme on Tuesday. She agreed to send the contract by Friday. Budget confirmed at $45,000.'),
    ('[TRIAGE]', 'Quick sync with dev team. Jake will fix the auth bug by EOD. Maria is starting the dashboard redesign next sprint.'),
    ('[EXPAND]', 'what about brian'),
    ('[ANSWER]', 'Question: What did we decide about pricing?\nMemories: [1] Meeting Mar 15 — decided to go freemium, generous limits. [2] Meeting Mar 22 — costs too high, reconsidering. [3] Meeting Mar 28 — reversed decision, going usage-based.'),
    ('[SUMMARIZE]', 'Board meeting recap: Revenue hit $2.1M ARR. Decided to raise Series A in Q3. Tom from Sequoia expressed interest. Need to prep deck by end of month.'),
    ('[SENTIMENT]', 'This is frustrating. We discussed this three times and nothing has changed. The deadline is tomorrow and we still dont have a plan.'),
    ('[IMPORTANCE]', 'Team standup: CSS fix deployed. Lunch order changed to Thai. Jenkins build is green.'),
]

for prefix, user_input in test_cases:
    task_name = prefix.strip('[]').lower()
    messages = [
        {'role': 'system', 'content': TASK_SYSTEM_PROMPTS[task_name]},
        {'role': 'user', 'content': f'{prefix} {user_input}'},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    print(f'\n--- {prefix} ---')
    print(f'Input:  {user_input[:80]}...' if len(user_input) > 80 else f'Input:  {user_input}')
    print(f'Output: {response}')

## Step 8: Save Model

In [ ]:
# Save LoRA adapters
model.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)
print(f'LoRA adapters saved to {OUTPUT_PATH}/')

# Save as GGUF for Ollama deployment (4-bit quantized)
GGUF_PATH = 'rabbit-v1-q4'
print(f'\nExporting GGUF (4-bit quantized)...')
model.save_pretrained_gguf(GGUF_PATH, tokenizer, quantization_method='q4_k_m')
print(f'GGUF saved to {GGUF_PATH}/')
print(f'\nThis is the file you deploy with Ollama.')

In [ ]:
# Download the GGUF model to your machine
import glob

gguf_files = glob.glob(f'{GGUF_PATH}/*.gguf')
if gguf_files:
    print(f'Downloading {gguf_files[0]}...')
    files.download(gguf_files[0])
else:
    print('GGUF file not found. Check the output above for errors.')
    print(f'Files in {GGUF_PATH}/:')
    !ls -la {GGUF_PATH}/

## Step 9: Push to Hugging Face (Optional, Private)

Keep the model private — this is proprietary.

In [ ]:
# Optional: push to private HF repo for easy deployment
# Uncomment and set your HF token

# HF_TOKEN = 'hf_...'  # Get from https://huggingface.co/settings/tokens
# HF_REPO = 'your-username/rabbit-v1'  # KEEP PRIVATE

# model.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
# tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
# print(f'Pushed to https://huggingface.co/{HF_REPO} (private)')

---

## What's Next

1. **Download the GGUF file** to your machine
2. **Deploy with Ollama:** `ollama create rabbit -f Modelfile`
3. **Test locally:** `ollama run rabbit '[INTENT] What did we discuss last week?'`
4. **Integrate with Reattend:** Point `OWN_MODEL_URL` to `http://localhost:11434`

Rabbit v1 is alive. 🐰